In [ ]:
import os 

os.environ["NEURON_EXPLAINER_API_KEY"] = "EMPTY"
os.environ["NEURON_EXPLAINER_API_BASE"] = "http://localhost:8000/v1" # Paste here your vLLM API base URL, with /v1 at the end

In [2]:
from neuron_explainer.activations.activation_records import calculate_max_activation
from neuron_explainer.activations.activations import ActivationRecordSliceParams, load_neuron
from neuron_explainer.explanations.calibrated_simulator import UncalibratedNeuronSimulator
from neuron_explainer.explanations.explainer import TokenActivationPairExplainer
from neuron_explainer.explanations.prompt_builder import PromptFormat
from neuron_explainer.explanations.scoring import simulate_and_score
from neuron_explainer.explanations.simulator import ExplanationNeuronSimulator, ExplanationTokenByTokenSimulator

In [3]:
EXPLAINER_MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
SIMULATOR_MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct"

In [4]:
from neuron_explainer.api_client import ApiClient

In [5]:
client = ApiClient(model_name="Qwen/Qwen3-Coder-30B-A3B-Instruct", max_concurrent=1)

In [6]:
test_response = await client.make_request(messages=[{"role": "user", "content": "What's up?"}], max_tokens=2)
print("Response:", test_response["choices"][0]["message"])

Response: {'role': 'assistant', 'content': 'Nothing much', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [], 'reasoning_content': None}


In [7]:
# Load a neuron record.
neuron_record = load_neuron(9, 6236)

In [8]:
# Grab the activation records we'll need.
slice_params = ActivationRecordSliceParams(n_examples_per_split=5)
train_activation_records = neuron_record.train_activation_records(
    activation_record_slice_params=slice_params
)
valid_activation_records = neuron_record.valid_activation_records(
    activation_record_slice_params=slice_params
)

In [9]:
# Generate an explanation for the neuron.
explainer = TokenActivationPairExplainer(
    model_name=EXPLAINER_MODEL_NAME,
    prompt_format=PromptFormat.HARMONY_V4,
    max_concurrent=1,
)

In [10]:
explanations = await explainer.generate_explanations(
    all_activation_records=train_activation_records,
    max_activation=calculate_max_activation(train_activation_records),
    num_samples=1,
)
assert len(explanations) == 1
explanation = explanations[0]
print(f"{explanation=}")

explanation=' phrases indicating repetition or multiple occurrences.'


In [11]:
# Simulate and score the explanation.
simulator = UncalibratedNeuronSimulator(
    ExplanationNeuronSimulator(
        SIMULATOR_MODEL_NAME,
        explanation,
        max_concurrent=1,
        prompt_format=PromptFormat.INSTRUCTION_FOLLOWING,
    )
)
scored_simulation = await simulate_and_score(simulator, valid_activation_records)
print(f"score={scored_simulation.get_preferred_score():.2f}")

score=0.08
